#### SET UP

In [1]:
import pandas as pd
import sys
sys.path.append("..")

from src.data.ingestion import load_raw_data

df = load_raw_data()
print("Shape:", df.shape)

c:\Users\meenakshi\Documents\readmission-poc\venv\Lib\site-packages\ucimlrepo\fetch.py:97: DtypeWarning: Columns (0: payer_code) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(data_url)


Shape: (101766, 50)


In [2]:
from src.data.validation import validate_raw_data

validate_raw_data(df)
print("Validation passed.")

Validation passed.


In [3]:
from src.data.cleaning import clean_data

df_audited = clean_data(df)

print("Shape:", df_audited.shape)
print("Y distribution:\n", df_audited["Y"].value_counts(normalize=True))

Shape: (99340, 48)
Y distribution:
 Y
0    0.886108
1    0.113892
Name: proportion, dtype: float64


In [4]:
from src.data.splitting import split_by_patient

train_df, test_df = split_by_patient(df_audited)

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)
print("Train patients:", train_df["patient_nbr"].nunique())
print("Test patients:", test_df["patient_nbr"].nunique())

overlap = set(train_df["patient_nbr"]) & set(test_df["patient_nbr"])
print("Patient overlap between train/test:", len(overlap))

Train shape: (79567, 48)
Test shape: (19773, 48)
Train patients: 55989
Test patients: 13998
Patient overlap between train/test: 0


In [5]:
#from src.data.splitting import split_by_patient

#train_df, test_df = split_by_patient(df_audited)

#PROCESSED_DIR = Path("../data/processed")
#train_df.to_csv(PROCESSED_DIR / "train.csv", index=False)
#test_df.to_csv(PROCESSED_DIR / "test.csv", index=False)

#print(f"Saved train.csv: {train_df.shape}")
#print(f"Saved test.csv:  {test_df.shape}")

In [6]:
print(train_df["gender"].unique())
print(train_df["age"].unique())

<ArrowStringArray>
['Female', 'Male']
Length: 2, dtype: str
<ArrowStringArray>
[  '[0-10)',  '[10-20)',  '[20-30)',  '[30-40)',  '[40-50)',  '[50-60)',
  '[60-70)',  '[80-90)', '[90-100)',  '[70-80)']
Length: 10, dtype: str


In [7]:
train_df["gender"].value_counts()

gender
Female    42728
Male      36839
Name: count, dtype: int64

In [8]:
from src.features.demographics import build_demographic_features

demo_features = build_demographic_features(train_df)
print(demo_features.shape)
demo_features.head()

(79567, 9)


,race_AfricanAmerican,race_Asian,race_Caucasian,race_Hispanic,race_Other,race_nan,gender_Female,gender_Male,age_midpoint
0,False,False,True,False,False,False,True,False,5
1,False,False,True,False,False,False,True,False,15
2,True,False,False,False,False,False,True,False,25
3,False,False,True,False,False,False,False,True,35
4,False,False,True,False,False,False,False,True,45


In [9]:
train_df[["admission_type_id", "admission_source_id", "time_in_hospital",
          "num_lab_procedures", "num_procedures", "num_medications",
          "number_outpatient", "number_emergency", "number_inpatient",
          "number_diagnoses"]].describe()

,admission_type_id,admission_source_id,time_in_hospital,num_lab_procedures,num_procedures,num_medications,number_outpatient,number_emergency,number_inpatient,number_diagnoses
count,79567.000000,79567.000000,79567.000000,79567.000000,79567.000000,79567.000000,79567.000000,79567.000000,79567.000000,79567.000000
mean,2.030113,5.724082,4.385059,42.938555,1.329810,15.997851,0.364988,0.200108,0.634710,7.402227
std,1.446087,4.057706,2.967360,19.569284,1.700907,8.119249,1.246746,0.975390,1.260413,1.941983
min,1.000000,1.000000,1.000000,1.000000,0.000000,1.000000,0.000000,0.000000,0.000000,1.000000
25%,1.000000,1.000000,2.000000,31.000000,0.000000,10.000000,0.000000,0.000000,0.000000,6.000000
50%,1.000000,7.000000,4.000000,44.000000,1.000000,15.000000,0.000000,0.000000,0.000000,8.000000
75%,3.000000,7.000000,6.000000,57.000000,2.000000,20.000000,0.000000,0.000000,1.000000,9.000000
max,8.000000,25.000000,14.000000,132.000000,6.000000,81.000000,42.000000,76.000000,21.000000,16.000000


In [10]:
from src.features.encounter import build_encounter_features

encounter_features = build_encounter_features(train_df)
print(encounter_features.shape)
encounter_features.head()

(79567, 33)


,time_in_hospital,num_lab_procedures,num_procedures,num_medications,number_outpatient,number_emergency,number_inpatient,number_diagnoses,admission_type_id_1,admission_type_id_2,...,admission_source_id_20,admission_source_id_22,admission_source_id_25,admission_source_id_3,admission_source_id_4,admission_source_id_5,admission_source_id_6,admission_source_id_7,admission_source_id_8,admission_source_id_9
0,1,41,0,1,0,0,0,1,False,False,...,False,False,False,False,False,False,False,False,False,False
1,3,59,0,18,0,0,0,9,True,False,...,False,False,False,False,False,False,False,True,False,False
2,2,11,5,13,2,0,1,6,True,False,...,False,False,False,False,False,False,False,True,False,False
3,2,44,1,16,0,0,0,7,True,False,...,False,False,False,False,False,False,False,True,False,False
4,1,51,0,8,0,0,0,5,True,False,...,False,False,False,False,False,False,False,True,False,False


In [11]:
from src.features.encounter import build_encounter_features

train_enc = build_encounter_features(train_df)
test_enc = build_encounter_features(test_df)

print("Train columns:", train_enc.shape[1])
print("Test columns:", test_enc.shape[1])

only_in_train = set(train_enc.columns) - set(test_enc.columns)
only_in_test = set(test_enc.columns) - set(train_enc.columns)
print("Columns only in train:", only_in_train)
print("Columns only in test:", only_in_test)

Train columns: 33
Test columns: 28
Columns only in train: {'admission_source_id_10', 'admission_source_id_14', 'admission_source_id_13', 'admission_source_id_25', 'admission_source_id_22'}
Columns only in test: set()


In [12]:
train_enc = build_encounter_features(train_df)                              # derives columns from train
test_enc = build_encounter_features(test_df, reference_columns=train_enc.columns.tolist())  # forced to match

print("Train columns:", train_enc.shape[1])
print("Test columns:", test_enc.shape[1])
print("Columns match:", list(train_enc.columns) == list(test_enc.columns))

Train columns: 33
Test columns: 33
Columns match: True


In [13]:
from src.features.demographics import build_demographic_features

train_demo = build_demographic_features(train_df)
test_demo = build_demographic_features(test_df, reference_columns=train_demo.columns.tolist())

print("Train columns:", train_demo.shape[1])
print("Test columns:", test_demo.shape[1])
print("Columns match:", list(train_demo.columns) == list(test_demo.columns))

Train columns: 9
Test columns: 9
Columns match: True


In [14]:
train_df["diag_1"].value_counts().head(10)

diag_1
428    5340
414    5239
786    3208
410    2755
486    2728
427    2141
491    1837
715    1729
682    1659
780    1594
Name: count, dtype: int64

In [15]:
from src.features.diagnoses import map_icd9_to_category

print(map_icd9_to_category("428"))     # expect Circulatory (actually... check this one carefully)
print(map_icd9_to_category("250.01"))  # expect Diabetes
print(map_icd9_to_category("786"))     # expect Respiratory
print(map_icd9_to_category("V27"))     # expect Other
print(map_icd9_to_category(None))      # expect Missing

Circulatory
Diabetes
Respiratory
Other
Missing


In [16]:
from src.features.diagnoses import build_diagnosis_features

train_diag = build_diagnosis_features(train_df)
test_diag = build_diagnosis_features(test_df)

print("Train shape:", train_diag.shape)
print("Test shape:", test_diag.shape)
print("Columns match:", list(train_diag.columns) == list(test_diag.columns))

for slot in ["diag_1", "diag_2", "diag_3"]:
    one_per_row = train_diag.filter(like=f"{slot}_").sum(axis=1).eq(1).all()
    print(f"{slot}: exactly one category per row ->", one_per_row)

Train shape: (79567, 30)
Test shape: (19773, 30)
Columns match: True
diag_1: exactly one category per row -> True
diag_2: exactly one category per row -> True
diag_3: exactly one category per row -> True


In [17]:
drug_cols = [
    "metformin", "repaglinide", "nateglinide", "chlorpropamide", "glimepiride",
    "acetohexamide", "glipizide", "glyburide", "tolbutamide", "pioglitazone",
    "rosiglitazone", "acarbose", "miglitol", "troglitazone", "tolazamide",
    "insulin", "glyburide-metformin", "glipizide-metformin",
    "glimepiride-pioglitazone", "metformin-rosiglitazone", "metformin-pioglitazone",
]

summary = pd.DataFrame({
    "pct_No": (train_df[drug_cols] == "No").mean(),
    "n_unique": train_df[drug_cols].nunique(),
}).sort_values("pct_No", ascending=False)

summary

,pct_No,n_unique
acetohexamide,0.999987,2
metformin-pioglitazone,0.999987,2
glimepiride-pioglitazone,0.999987,2
metformin-rosiglitazone,0.999975,2
troglitazone,0.999962,2
glipizide-metformin,0.999837,2
tolbutamide,0.999811,2
miglitol,0.999724,2
tolazamide,0.999623,3
chlorpropamide,0.999145,4


In [18]:
train_df["insulin"].value_counts()

insulin
No        37122
Steady    24235
Down       9473
Up         8737
Name: count, dtype: int64

In [19]:
from src.features.medications import build_medication_features, KEPT_DRUGS

train_med = build_medication_features(train_df)
test_med = build_medication_features(test_df)

print("Train shape:", train_med.shape)
print("Test shape:", test_med.shape)
print("Columns match:", list(train_med.columns) == list(test_med.columns))

for drug in KEPT_DRUGS:
    ok = train_med.filter(like=f"{drug}_").sum(axis=1).eq(1).all()
    print(f"{drug}: exactly one status per row ->", ok)

print(train_med[["n_drugs_prescribed", "n_drugs_changed"]].describe())

Train shape: (79567, 36)
Test shape: (19773, 36)
Columns match: True
metformin: exactly one status per row -> True
repaglinide: exactly one status per row -> True
glimepiride: exactly one status per row -> True
glipizide: exactly one status per row -> True
glyburide: exactly one status per row -> True
pioglitazone: exactly one status per row -> True
rosiglitazone: exactly one status per row -> True
insulin: exactly one status per row -> True
       n_drugs_prescribed  n_drugs_changed
count        79567.000000     79567.000000
mean             1.188470         0.285721
std              0.925211         0.487101
min              0.000000         0.000000
25%              1.000000         0.000000
50%              1.000000         0.000000
75%              2.000000         1.000000
max              6.000000         4.000000


In [20]:
no_drugs = train_med["n_drugs_prescribed"] == 0
no_flag = train_med["diabetes_med_flag"] == 0
print("Agreement between the two measures:", (no_drugs == no_flag).mean())
print("Rows where changed > prescribed:", (train_med["n_drugs_changed"] > train_med["n_drugs_prescribed"]).sum())

Agreement between the two measures: 1.0
Rows where changed > prescribed: 0


PAYER.PY


In [21]:
train_df["payer_code"].value_counts(dropna=False)

payer_code
NaN    31566
MC     25054
HM      5010
SP      3941
BC      3679
MD      2788
CP      1949
UN      1947
CM      1506
OG       804
DM       453
PO       453
WC       110
CH       108
OT        79
MP        73
SI        46
FR         1
Name: count, dtype: int64

In [22]:
from src.features.payer import build_payer_features

train_payer = build_payer_features(train_df)
test_payer = build_payer_features(test_df)

print("Train shape:", train_payer.shape)
print("Test shape:", test_payer.shape)
print("Columns match:", list(train_payer.columns) == list(test_payer.columns))
print("Exactly one category per row:", train_payer.sum(axis=1).eq(1).all())
print(train_payer.sum().sort_values(ascending=False))

Train shape: (79567, 11)
Test shape: (19773, 11)
Columns match: True
Exactly one category per row: True
payer_Missing    31566
payer_MC         25054
payer_HM          5010
payer_SP          3941
payer_BC          3679
payer_MD          2788
payer_CP          1949
payer_UN          1947
payer_CM          1506
payer_Other       1323
payer_OG           804
dtype: int64


In [23]:
from src.features.build_features import build_features

X_train = build_features(train_df)
X_test = build_features(test_df, reference_columns=X_train.columns.tolist())

print("Train:", X_train.shape, " Test:", X_test.shape)
print("Columns match:", list(X_train.columns) == list(X_test.columns))

X_train_nopayer = build_features(train_df, include_payer=False)
print("Without payer:", X_train_nopayer.shape)

corr = X_train.corrwith(train_df["Y"]).abs().sort_values(ascending=False)
print("\nStrongest single-feature correlations with Y:")
print(corr.head(8))

constant = X_train.columns[X_train.nunique() == 1].tolist()
print("\nConstant columns in train:", constant)

Train: (79567, 119)  Test: (19773, 119)
Columns match: True
Without payer: (79567, 108)

Strongest single-feature correlations with Y:
number_inpatient    0.164417
number_emergency    0.058102
number_diagnoses    0.055835
time_in_hospital    0.048558
num_medications     0.040508
n_drugs_changed     0.034465
insulin_No          0.033818
insulin_Down        0.032623
dtype: float64

Constant columns in train: []


In [24]:
from src.data.versioning import save_dataset_version, load_dataset_version

version_dir = save_dataset_version(
    X_train, X_test,
    train_df["Y"], test_df["Y"],
    train_df["patient_nbr"], test_df["patient_nbr"],
    settings={
        "include_payer": True,
        "split": "GroupShuffleSplit by patient_nbr",
        "test_size": 0.2,
        "random_seed": 42,
        "note": "Drop columns starting with 'payer_' for the without-payer comparison.",
    },
)
print("Saved to:", version_dir)

data = load_dataset_version(version_dir)
print("Loaded OK. Version:", data["manifest"]["version"])
print("Train:", data["X_train"].shape, " Test:", data["X_test"].shape)
print("Positive rate train/test:", data["manifest"]["positive_rate"])

Saved to: C:\Users\meenakshi\Documents\readmission-poc\data\processed\v_86a646f1
Loaded OK. Version: 86a646f1
Train: (79567, 119)  Test: (19773, 119)
Positive rate train/test: {'train': 0.113992, 'test': 0.113488}
